# Kaggle submission: response length baseline
GROUP50_CENG313, LLM Classification Finetuning

In [ ]:
import glob
import json
import os

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

def find_data_dir(kaggle_root="/kaggle/input", local_dir="data"):
    """On Kaggle, look for train.csv anywhere under /kaggle/input; otherwise use the local data/ folder."""
    if os.path.isdir(kaggle_root):
        hits = sorted(glob.glob(os.path.join(kaggle_root, "**", "train.csv"), recursive=True))
        if hits:
            return os.path.dirname(hits[0])
        raise FileNotFoundError(
            f"No train.csv under {kaggle_root}. Add the competition data: right panel > Input > "
            "Add Input > Competition > LLM Classification Finetuning (join the competition first). "
            f"Files found: {os.listdir(kaggle_root)}"
        )
    return local_dir


DATA_DIR = find_data_dir()
print("Using data from:", DATA_DIR)

TARGET_COLS = ["winner_model_a", "winner_model_b", "winner_tie"]
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "test:", test.shape)

In [ ]:
def word_count(turns_json):
    """Total words in a JSON-encoded list of turns; null turns are skipped."""
    turns = json.loads(turns_json)
    return sum(len(turn.split()) for turn in turns if isinstance(turn, str))


def length_features(df):
    a_words = df["response_a"].map(word_count)
    b_words = df["response_b"].map(word_count)
    return pd.DataFrame({"a_words": a_words, "b_words": b_words, "word_diff": a_words - b_words})


X_train = length_features(train)
y_train = train[TARGET_COLS].to_numpy().argmax(axis=1)   # 0 = A wins, 1 = B wins, 2 = tie

model = make_pipeline(
    StandardScaler(),
    FunctionTransformer(csr_matrix),
    LogisticRegression(),
)
model.fit(X_train, y_train)
print("Model fitted on", len(X_train), "rows")

In [ ]:
test_proba = model.predict_proba(length_features(test))

submission = pd.DataFrame(test_proba, columns=TARGET_COLS)
submission.insert(0, "id", test["id"].to_numpy())
submission.to_csv("submission.csv", index=False)

assert np.allclose(submission[TARGET_COLS].sum(axis=1), 1.0)
print("Saved submission.csv with", len(submission), "rows")
submission.head()